
# Spectral Rolloff Extraction

## Aim
This notebook extracts **spectral rolloff** from preprocessed beehive recordings and aligns it with environmental variables such as temperature and humidity.

Spectral rolloff represents the frequency below which a certain percentage (typically 85%) of the total spectral energy is contained.



## Notes

Because preprocessing included normalisation, rolloff values should be interpreted as **relative spectral distribution changes over time**.


In [ ]:

import os
from glob import glob
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from sklearn.preprocessing import MinMaxScaler


## Configuration

In [ ]:

TARGET_SR = 16000
N_FFT = 2048
HOP_SAMPLES = 16000
CHUNK_SECONDS = 3600

INPUT_FOLDER = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024"
OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "SR_features_refined")

ENV_PATH = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024\Env_logs\HaniBi_20240815T134229+0200.xlsx"

BASE_START_TIME = pd.Timestamp("2024-08-13 11:20:00")
START_SLICE = pd.Timestamp("2024-08-13 11:20:00")
END_SLICE = pd.Timestamp("2024-08-15 13:04:41")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


## Helper functions

In [ ]:

def extract_spectral_rolloff(audio_path, start_time, n_fft=N_FFT, hop_length=HOP_SAMPLES):
    y, sr = librosa.load(audio_path, sr=None)

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length
    ).squeeze()

    times = librosa.frames_to_time(
        np.arange(len(rolloff)),
        sr=sr,
        hop_length=hop_length
    )

    timestamps = [start_time + timedelta(seconds=float(t)) for t in times]

    return pd.DataFrame({
        "timestamp": timestamps,
        "spectral_rolloff": rolloff
    })


def load_env(env_path, start_slice, end_slice):
    env_df = pd.read_excel(env_path)
    env_df = env_df.rename(columns={"Date": "timestamp"})
    env_df["timestamp"] = pd.to_datetime(env_df["timestamp"])

    env_df = env_df[(env_df["timestamp"] >= start_slice) & (env_df["timestamp"] <= end_slice)]

    env_df = env_df[["timestamp", "Temperatura (°C)", "Wilgotność (%)"]]

    env_df = env_df.set_index("timestamp").resample("1S").interpolate().reset_index()

    return env_df


def build_start_times(files, base_start):
    durations = [librosa.get_duration(path=f) for f in files]

    times = []
    elapsed = 0
    for d in durations:
        times.append(base_start + timedelta(seconds=float(elapsed)))
        elapsed += d
    return times


def process_all(input_folder, output_folder, env_df, base_start):
    files = sorted(glob(os.path.join(input_folder, "*.wav")))
    starts = build_start_times(files, base_start)

    logs = []

    for f, st in zip(files, starts):
        name = os.path.basename(f)

        try:
            df = extract_spectral_rolloff(f, st)

            merged = pd.merge_asof(
                df.sort_values("timestamp"),
                env_df.sort_values("timestamp"),
                on="timestamp",
                direction="nearest"
            )

            out_path = os.path.join(output_folder, name.replace(".wav", "_rolloff.csv"))
            merged.to_csv(out_path, index=False)

            logs.append({"file": name, "status": "ok"})

        except Exception as e:
            logs.append({"file": name, "status": "fail", "error": str(e)})

    return pd.DataFrame(logs)


## Run extraction

In [ ]:

env_df = load_env(ENV_PATH, START_SLICE, END_SLICE)

log = process_all(INPUT_FOLDER, OUTPUT_FOLDER, env_df, BASE_START_TIME)

log


## Combine results

In [ ]:

files = glob(os.path.join(OUTPUT_FOLDER, "*_rolloff.csv"))

combined = pd.concat([pd.read_csv(f, parse_dates=["timestamp"]) for f in files])

combined = combined.dropna()

combined.head()


## Normalisation

In [ ]:

scaler = MinMaxScaler()

norm = scaler.fit_transform(
    combined[["spectral_rolloff", "Temperatura (°C)", "Wilgotność (%)"]]
)

norm_df = pd.DataFrame(norm, columns=["rolloff", "temp", "hum"])
norm_df["timestamp"] = combined["timestamp"]
norm_df["elapsed"] = (norm_df["timestamp"] - norm_df["timestamp"].iloc[0]).dt.total_seconds()


## Plot

In [ ]:

plt.figure(figsize=(14,5))
plt.plot(norm_df["timestamp"], norm_df["rolloff"], label="Rolloff")
plt.plot(norm_df["timestamp"], norm_df["temp"], label="Temp")
plt.plot(norm_df["timestamp"], norm_df["hum"], label="Hum")
plt.legend()
plt.show()



## Interpretation

Spectral rolloff indicates how much high-frequency energy is present.

- higher rolloff → more high-frequency content (possible agitation or disturbance)
- lower rolloff → energy concentrated in lower frequencies (stable buzzing)

This feature helps identify shifts in hive acoustic behaviour over time.
